In [2]:
import pandas as pd
import numpy as np

In [23]:
df = pd.read_excel("a112a.xlsx")

In [24]:
df.shape

(72741, 28)

In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72741 entries, 0 to 72740
Data columns (total 32 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   Category                      72741 non-null  object        
 1   City                          72741 non-null  object        
 2   Month                         72741 non-null  object        
 3   Sub Category                  72741 non-null  object        
 4   Date of Discovery             72741 non-null  datetime64[ns]
 5   Day                           72741 non-null  int64         
 6   Year                          72741 non-null  int64         
 7   Company name                  72734 non-null  object        
 8   Brand name                    72741 non-null  object        
 9   Variant/Flavour               72381 non-null  object        
 10  Type of Change                72723 non-null  object        
 11  Offer on pack               

In [26]:
numeric_cols = [
    "MRP Current", "Retailer Price",
    "Price Discount", "Absolute Discount",
    "Effective Price Per Unit"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")


In [27]:
df["Discount %"] = (df["Absolute Discount"] / df["MRP Current"]) * 100
df["Retail Margin"] = df["MRP Current"] - df["Retailer Price"]
df["Price Efficiency"] = df["Effective Price Per Unit"]

In [28]:
pack_cols = [
    "No. units in a pack",
    "No. of packs in strip",
    "No. pouches in Carton"
]

for col in pack_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")




In [9]:
# -----------------------------
# SKU / Packaging Intelligence
# -----------------------------
df["Total Units"] = (
    df["No. units in a pack"].fillna(1) *
    df["No. of packs in strip"].fillna(1) *
    df["No. pouches in Carton"].fillna(1)
)

df["Price per Unit"] = df["Retailer Price"] / df["Total Units"]

df["SKU Changed"] = df["Type of Change"].astype(str).str.contains("size", case=False, na=False)

# Sort for time-based calculations
df = df.sort_values(["Brand name", "Date of Discovery"])

df["Shrinkflation Flag"] = (
    (df["SKU Changed"] == True) &
    (df.groupby("Brand name")["Price per Unit"].pct_change() > 0)
)
df["YearMonth"] = df["Date of Discovery"].dt.to_period("M")

df["Price Change %"] = (
    df.groupby("Brand name")["MRP Current"].pct_change() * 100
)

TypeError: 'values' is not ordered, please explicitly specify the categories order by passing in a categories argument.

In [29]:
# -----------------------------
# Time-Based Features
# -----------------------------
'''

'''
# -----------------------------
# Market Intelligence
# -----------------------------
city_price = df.groupby("City")["MRP Current"].mean().sort_values(ascending=False)
brand_activity = df["Brand name"].value_counts()

# -----------------------------
# Offer Intelligence
# -----------------------------
df["Has Offer"] = df["Offer on pack"].notna()
offer_impact = df.groupby("Has Offer")["Discount %"].mean()

## 3. Cleaning Subcategories

In [30]:
print(df['Category'].unique())

['Breakfast Cereals (Fills/Cornflakes/Oats/Muesli)' 'Packaged Atta'
 'Packaged atta' 'Bakery (Rusk/Toast/Cakes/Khari)'
 'Salty Snacks (Western/Traditional)' 'Biscuits' 'Confectionery'
 'Breakfast Cereals (Fills/Cornflakes/Oats/Muesli/ Quinoa)']


In [31]:
print(df['Sub Category '].unique())

['Whole Wheat Flakes' 'Wheat ' 'Wheat' 'Waffles' 'Wafers' 'Cookies'
 'Wafer Roll' 'Non Salted' 'Salted' 'Marie' 'Cream' 'Wafer Chocolate'
 'Digestive ' 'Wafer Biscuits' 'Digestive' 'Sweet & Salted' 'Top'
 'Cracker' 'Cookies ' 'Milk' 'Sweet & Salted ' 'Cream ' 'Digestive Cream'
 'Bhakarwadi' 'Wada Peeth' 'Udad Dal' 'Truffle' 'Toffee'
 'Thalipeeth Bhajani' 'Swiss Roll' 'Glucose' 'Non Salted ' 'Mixture'
 'Stix  ' 'Stix' 'Cake' 'Soybean Atta' 'Soft Candy' 'Chewy Candy'
 'Soft candy' 'soft candy' 'Snack Bar' 'Slice Cake' 'slice Cake'
 'Slice cake' 'Singhada' 'Sev' 'Sattu ' 'Sattu' 'Salted ' 'Rusk / Toast'
 'Rusk / Toast ' 'Rusk / toast' 'Roll Cake' 'Rice ' 'Rice' 'Rajgara '
 'Rajgara' 'Ragi' 'Quick Bread' 'Puttupodi' 'Puffs' 'Pudding ' 'Pudding'
 'pudding' 'Protein Bar' 'Protein bar' 'Praline' 'Bridges' 'Pie ' 'Pie'
 'Chana & Nuts' 'Papadi' 'papadi' 'Panned Chocolate' 'Panned chocolate'
 'Pakoda' 'Oats ' 'Oats' 'Nutrition Bar' 'Nutrition bar' 'Cheeseling'
 'Nachni (Ragi)' 'Murukku' 'MuruKKu

In [32]:
print("\n Snacks")
print(df[df['Category'] == 'Salty Snacks (Western/Traditional)']['Sub Category '].unique())
print("\n Breakfast")
print(df[df['Category'] == 'Breakfast Cereals (Fills/Cornflakes/Oats/Muesli)']['Sub Category '].unique())
print("\n Confectionery")
print(df[df['Category'] == 'Confectionery']['Sub Category '].unique())
print("\n Bakery")
print(df[df['Category'] == 'Bakery (Rusk/Toast/Cakes/Khari)']['Sub Category '].unique())
print("\n Biscuits")
print(df[df['Category'] == 'Biscuits']['Sub Category '].unique())


 Snacks
['Wafers' 'Bhakarwadi' 'Mixture' 'Stix  ' 'Stix' 'Sev' 'Bridges'
 'Chana & Nuts' 'Papadi' 'papadi' 'Pakoda' 'Murukku' 'MuruKKu' 'Mixture '
 'MIxture' 'Mattar' 'Makhana ' 'Makhana' 'Kachori' 'Kara Boondi' 'Chiwda'
 'Assorted Snacks' 'Gathiya' 'Dal    ' 'Dal' 'Chips  ' 'Chips ' 'Chips'
 'chips' 'Chakali' 'Bridges ' 'Bridges  ' 'Boondi ' 'Boondi' 'Bhujia   '
 'Bhujia ' 'Bhujia' 'Assorted Snacks ' 'Assorted snacks' 'Broad Beans'
 'Nuts' ' Pakoda' ' Nuts' 'Chana' 'Bakarwadi' 'Chana ' 'Nuts ' 'Ghatiya'
 'Chips  ( Crisps)' 'Chips (Crisps)' ' Chips ( Crisps) ' 'Matar' 'Nut'
 'Crisps ( Chips)' 'Wafer ' 'Mixed Pulses' 'Green peas & Nuts'
 'Baked Chips' 'Wafer' 'Chips ( Crisps)' 'Chakli' 'BhakarWadi' 'Stix '
 'Seeds' 'Sweet Butter Sticks' 'Chips  ( Crisps) ' 'Chips   (Crisps )'
 'Chips  (Crisps)' 'Chips ( Crisps) ' 'Chips  (crisps)' 'Chips   (Crisps)'
 'Party Mix' ' Chips(Crisps )' 'Millet Mixture' 'Millet Mixture ' 'Donuts']

 Breakfast
['Whole Wheat Flakes' 'Puffs' 'Oats ' 'Oats' 'Mult

### Salty

In [33]:
# -----------------------------
# First: Basic whitespace cleanup (IMPORTANT)
# -----------------------------
df['Sub Category '] = (
    df['Sub Category ']
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# -----------------------------
# Bhakarwadi
# -----------------------------
values_to_replace_bhakarwadi = ['Bakarwadi', 'BhakarWadi']
df.loc[df['Sub Category '].isin(values_to_replace_bhakarwadi), 'Sub Category '] = 'Bhakarwadi'

# -----------------------------
# Mixture
# -----------------------------
values_to_replace_mixture = ['MIxture', 'Mixture ']
df.loc[df['Sub Category '].isin(values_to_replace_mixture), 'Sub Category '] = 'Mixture'

# -----------------------------
# Papadi
# -----------------------------
values_to_replace_papadi = ['papadi']
df.loc[df['Sub Category '].isin(values_to_replace_papadi), 'Sub Category '] = 'Papadi'

# -----------------------------
# Murukku
# -----------------------------
values_to_replace_murukku = ['MuruKKu']
df.loc[df['Sub Category '].isin(values_to_replace_murukku), 'Sub Category '] = 'Murukku'

# -----------------------------
# Mattar
# -----------------------------
values_to_replace_mattar = ['Matar']
df.loc[df['Sub Category '].isin(values_to_replace_mattar), 'Sub Category '] = 'Mattar'

# -----------------------------
# Assorted Snacks
# -----------------------------
values_to_replace_assorted = ['Assorted snacks', 'Assorted Snacks ']
df.loc[df['Sub Category '].isin(values_to_replace_assorted), 'Sub Category '] = 'Assorted Snacks'

# -----------------------------
# Gathiya
# -----------------------------
values_to_replace_gathiya = ['Ghatiya']
df.loc[df['Sub Category '].isin(values_to_replace_gathiya), 'Sub Category '] = 'Gathiya'

# -----------------------------
# Chips (Crisps)
# -----------------------------
values_to_replace_chips = [
    'chips', 'Chips', 'Chips ', 'Chips  ',
    'Chips (Crisps)', 'Chips ( Crisps)', 'Chips  ( Crisps)',
    'Chips   (Crisps )', 'Chips  (Crisps)', 'Chips  (crisps)',
    'Chips   (Crisps)', 'Chips(Crisps )', ' Chips ( Crisps) ',
    ' Chips(Crisps )', 'Crisps ( Chips)'
]
df.loc[df['Sub Category '].isin(values_to_replace_chips), 'Sub Category '] = 'Chips (Crisps)'

# -----------------------------
# Chakli
# -----------------------------
values_to_replace_chakli = ['Chakali']
df.loc[df['Sub Category '].isin(values_to_replace_chakli), 'Sub Category '] = 'Chakli'

# -----------------------------
# Nuts
# -----------------------------
values_to_replace_nuts = ['Nut', ' Nuts', 'Nuts ', ' Nuts ']
df.loc[df['Sub Category '].isin(values_to_replace_nuts), 'Sub Category '] = 'Nuts'

# -----------------------------
# Wafers
# -----------------------------
values_to_replace_wafers = ['Wafer', 'Wafer ']
df.loc[df['Sub Category '].isin(values_to_replace_wafers), 'Sub Category '] = 'Wafers'

# -----------------------------
# Bridges
# -----------------------------
values_to_replace_bridges = ['Bridges ', 'Bridges  ']
df.loc[df['Sub Category '].isin(values_to_replace_bridges), 'Sub Category '] = 'Bridges'

# -----------------------------
# Boondi
# -----------------------------
values_to_replace_boondi = ['Boondi ', 'Boondi']
df.loc[df['Sub Category '].isin(values_to_replace_boondi), 'Sub Category '] = 'Boondi'

# -----------------------------
# Bhujia
# -----------------------------
values_to_replace_bhujia = ['Bhujia ', 'Bhujia   ']
df.loc[df['Sub Category '].isin(values_to_replace_bhujia), 'Sub Category '] = 'Bhujia'

# -----------------------------
# Dal
# -----------------------------
values_to_replace_dal = ['Dal    ', 'Dal']
df.loc[df['Sub Category '].isin(values_to_replace_dal), 'Sub Category '] = 'Dal'

# -----------------------------
# Pakoda
# -----------------------------
values_to_replace_pakoda = [' Pakoda']
df.loc[df['Sub Category '].isin(values_to_replace_pakoda), 'Sub Category '] = 'Pakoda'

# -----------------------------
# Chana
# -----------------------------
values_to_replace_chana = ['Chana ', 'Chana']
df.loc[df['Sub Category '].isin(values_to_replace_chana), 'Sub Category '] = 'Chana'

# -----------------------------
# Stix
# -----------------------------
values_to_replace_stix = ['Stix ', 'Stix  ']
df.loc[df['Sub Category '].isin(values_to_replace_stix), 'Sub Category '] = 'Stix'

# -----------------------------
# Makhana
# -----------------------------
values_to_replace_makhana = ['Makhana ']
df.loc[df['Sub Category '].isin(values_to_replace_makhana), 'Sub Category '] = 'Makhana'

# -----------------------------
# Millet Mixture
# -----------------------------
values_to_replace_millet_mix = ['Millet Mixture ']
df.loc[df['Sub Category '].isin(values_to_replace_millet_mix), 'Sub Category '] = 'Millet Mixture'

# -----------------------------
# Final Check
# -----------------------------
print(df[df['Category'] == 'Salty Snacks (Western/Traditional)']['Sub Category '].unique())

['Wafers' 'Bhakarwadi' 'Mixture' 'Stix' 'Sev' 'Bridges' 'Chana & Nuts'
 'Papadi' 'Pakoda' 'Murukku' 'Mattar' 'Makhana' 'Kachori' 'Kara Boondi'
 'Chiwda' 'Assorted Snacks' 'Gathiya' 'Dal' 'Chips (Crisps)' 'Chakli'
 'Boondi' 'Bhujia' 'Broad Beans' 'Nuts' 'Chana' 'Mixed Pulses'
 'Green peas & Nuts' 'Baked Chips' 'Seeds' 'Sweet Butter Sticks'
 'Chips (Crisps )' 'Chips (crisps)' 'Party Mix' 'Millet Mixture' 'Donuts']


### Breakfast

In [34]:
# -----------------------------
# Basic Cleaning First (IMPORTANT)
# -----------------------------
df['Sub Category '] = (
    df['Sub Category ']
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# -----------------------------
# Oats / Millet-Oats combos
# -----------------------------
values_to_replace_oats = [
    'Millets+Oats', 'Millets + Oats', 'Millets & Oats',
    'Millet+Oats', 'Millet + Oats', 'Multigrain Oats'
]
df.loc[df['Sub Category '].isin(values_to_replace_oats), 'Sub Category '] = 'Millets + Oats'

# -----------------------------
# Cornflakes / Cornflake
# -----------------------------
values_to_replace_cornflakes = [
    'Cornflakes', 'CornFlakes', 'Cornflake',
    'Mulitgrain Cornflakes', 'Multigrain Cornflakes',
    'Multigrain Cornflake'
]
df.loc[df['Sub Category '].isin(values_to_replace_cornflakes), 'Sub Category '] = 'Mulitgrain Cornflakes'

# -----------------------------
# Multigrain Choco Flakes
# -----------------------------
values_to_replace_choco_flakes = [
    'Multigrain Choco Flakes', 'Multigrain choco Flakes',
    'Multi Millet Choco Flakes', 'Multigrain Chocos Flakes',
    'Multigrain ChocoFlakes'
]
df.loc[df['Sub Category '].isin(values_to_replace_choco_flakes), 'Sub Category '] = 'Multigrain Choco Flakes'

# -----------------------------
# Fruit Loops / Rings
# -----------------------------
values_to_replace_loops = [
    'Multigrain Fruit Loops', 'Multigrain Fruit Loop',
    'Multigrain Choco Loops', 'Multigrain Choco Rings'
]
df.loc[df['Sub Category '].isin(values_to_replace_loops), 'Sub Category '] = 'Multigrain Loops'

# -----------------------------
# Flakes (generic)

# -----------------------------
# Granola / Muesli
# -----------------------------
values_to_replace_granola = [
    'Granola', 'Granola Museli'
]
df.loc[df['Sub Category '].isin(values_to_replace_granola), 'Sub Category '] = 'Granola'

values_to_replace_muesli = ['Muesli']
df.loc[df['Sub Category '].isin(values_to_replace_muesli), 'Sub Category '] = 'Muesli'

# -----------------------------
# Millet-based
# -----------------------------
values_to_replace_millet = [
    'Millet Flakes', 'Whole Grain Millet'
]
df.loc[df['Sub Category '].isin(values_to_replace_millet), 'Sub Category '] = 'Millet Flakes'

# -----------------------------
# Quinoa
# -----------------------------
values_to_replace_quinoa = ['Quinoa']
df.loc[df['Sub Category '].isin(values_to_replace_quinoa), 'Sub Category '] = 'Quinoa'

print(df[df['Category'] == 'Breakfast Cereals (Fills/Cornflakes/Oats/Muesli)']['Sub Category '].unique())


['Whole Wheat Flakes' 'Puffs' 'Oats' 'Multigrain Choco Flakes'
 'Multigrain Corn Flakes' 'Multigrain Loops' 'Muesli' 'Millet Flakes'
 'Granola' 'Fills' 'Mulitgrain Cornflakes' 'Millets + Oats' 'Granola Bar'
 'Granola Bars' 'Oats n Multigrain Flakes' 'Multigrain Flakes'
 'Wheat Choco Flakes' 'Wheat Flakes' 'Quinoa']


### Biscuits

In [35]:
# -----------------------------
# Basic Cleaning First
# -----------------------------
df['Sub Category '] = (
    df['Sub Category ']
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# -----------------------------
# Salted / Non-Salted
# -----------------------------
values_to_replace_salted = ['Salted', 'salted']
df.loc[df['Sub Category '].isin(values_to_replace_salted), 'Sub Category '] = 'Salted'

values_to_replace_nonsalted = ['Non Salted', 'Non salted']
df.loc[df['Sub Category '].isin(values_to_replace_nonsalted), 'Sub Category '] = 'Non Salted'

# -----------------------------
# Sweet & Salted
# -----------------------------
values_to_replace_sweet_salted = [
    'Sweet & Salted', 'Sweet & salted'
]
df.loc[df['Sub Category '].isin(values_to_replace_sweet_salted), 'Sub Category '] = 'Sweet & Salted'

# -----------------------------
# Digestive
# -----------------------------
values_to_replace_digestive = [
    'Digestive', 'Digetives', 'Digestive Cookies'
]
df.loc[df['Sub Category '].isin(values_to_replace_digestive), 'Sub Category '] = 'Digestive'

values_to_replace_digestive_cream = ['Digestive Cream']
df.loc[df['Sub Category '].isin(values_to_replace_digestive_cream), 'Sub Category '] = 'Digestive Cream'

# -----------------------------
# Cream Biscuits
# -----------------------------
values_to_replace_cream = [
    'Cream', 'Open Cream Biscuit', 'Creamy Fruit Dessert'
]
df.loc[df['Sub Category '].isin(values_to_replace_cream), 'Sub Category '] = 'Cream Biscuit'

# -----------------------------
# Crackers
# -----------------------------
values_to_replace_crackers = [
    'Cracker', 'Crackers', 'Cream Cracker',
    'Sweet & Salted Cracker', 'Sweet & Salt Cracker',
]
df.loc[df['Sub Category '].isin(values_to_replace_crackers), 'Sub Category '] = 'Crackers'

# -----------------------------
# Wafer Biscuits
# -----------------------------
values_to_replace_wafer = [
    'Wafer Biscuits', 'Wafers Biscuits', 'Wafer Biscuit',
]
df.loc[df['Sub Category '].isin(values_to_replace_wafer), 'Sub Category '] = 'Wafer Biscuit'

# -----------------------------
# Cookies
# -----------------------------
values_to_replace_cookies = ['Cookies', 'Cookie Tart']
df.loc[df['Sub Category '].isin(values_to_replace_cookies), 'Sub Category '] = 'Cookies'

# -----------------------------
# Chocolate Biscuits
# -----------------------------
values_to_replace_choco = ['Chocolate Biscuits']
df.loc[df['Sub Category '].isin(values_to_replace_choco), 'Sub Category '] = 'Chocolate Biscuit'

# -----------------------------
# Glucose / Marie / Milk
# -----------------------------
values_to_replace_glucose = ['Glucose']
df.loc[df['Sub Category '].isin(values_to_replace_glucose), 'Sub Category '] = 'Glucose Biscuit'

values_to_replace_marie = ['Marie']
df.loc[df['Sub Category '].isin(values_to_replace_marie), 'Sub Category '] = 'Marie Biscuit'

values_to_replace_milk = ['Milk']
df.loc[df['Sub Category '].isin(values_to_replace_milk), 'Sub Category '] = 'Milk Biscuit'

# -----------------------------
# Chips Biscuit (typo)
# -----------------------------
values_to_replace_chip_biscuit = ['Chip Biscuits', 'Chip Biscuit']
df.loc[df['Sub Category '].isin(values_to_replace_chip_biscuit), 'Sub Category '] = 'Chip Biscuit'

# -----------------------------
# TOP branding
# -----------------------------
values_to_replace_top = ['Top', 'TOP']
df.loc[df['Sub Category '].isin(values_to_replace_top), 'Sub Category '] = 'Top'

# -----------------------------
# Specialty / Health
# -----------------------------
values_to_replace_health = [
    'Fiber Rich Biscuit', 'Vegan Biscuit', 'Multigrain Millet'
]
df.loc[df['Sub Category '].isin(values_to_replace_health), 'Sub Category '] = 'Healthy Biscuit'



# -----------------------------
# Final Check
# -----------------------------
print(df[df['Category'] == 'Biscuits']['Sub Category '].unique())

['Cookies' 'Non Salted' 'Salted' 'Marie Biscuit' 'Cream Biscuit'
 'Digestive' 'Wafer Biscuit' 'Sweet & Salted' 'Top' 'Crackers'
 'Milk Biscuit' 'Digestive Cream' 'Glucose Biscuit' 'Cheeseling'
 'Chocolate Biscuit' 'Chip Biscuit' 'Wafer Roll Biscuits'
 'Healthy Biscuit' 'Savory-Sweet Sandwich Cracker'
 'Crunchy,Spice-Packed Caramelized Biscuit' 'Wafer Roll'
 'Wafer Roll Biscuit' 'Savoury Cracker Sandwich Biscuit'
 'Sandwich Cracker' 'Savory Cracker Sandwich Biscuit' 'Cream Roll'
 'Millet Wafer']


### Confectionery

In [36]:
# -----------------------------
# Basic Cleaning First (IMPORTANT)
# -----------------------------
df['Sub Category '] = (
    df['Sub Category ']
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# -----------------------------
# Candy (soft / chewy / hard / jelly)
# -----------------------------
values_to_replace_candy = [
    'Soft Candy', 'Soft candy', 'soft candy', 'Sofy Candy'
]
df.loc[df['Sub Category '].isin(values_to_replace_candy), 'Sub Category '] = 'Soft Candy'

values_to_replace_juju = [
    'Jubjubes', 'JuJubes'
]
df.loc[df['Sub Category '].isin(values_to_replace_juju), 'Sub Category '] = 'Jujubes'


# -----------------------------
# Toffee
# -----------------------------
values_to_replace_toffee = [
    'Toffee', 'Toffees', 'Chewy Toffee'
]
df.loc[df['Sub Category '].isin(values_to_replace_toffee), 'Sub Category '] = 'Toffee'

# -----------------------------
# Chocolate
# -----------------------------
values_to_replace_chocolate = [
    'Bar Chocolate', 'Bar chocolate',
    'Assorted Chocolate', 'Multigrain Chocolate'
]
df.loc[df['Sub Category '].isin(values_to_replace_chocolate), 'Sub Category '] = 'Bar Chocolate'

# -----------------------------
# Wafer / Wafer-based
# -----------------------------
values_to_replace_wafer = [
    'Wafer Roll', 'Wafer Chocolate', 'Millet Wafer',
    'Chocolate Wafer Bar'
]
df.loc[df['Sub Category '].isin(values_to_replace_wafer), 'Sub Category '] = 'Wafer Chocolate'

# -----------------------------
# Bars (ALL types)
# -----------------------------
'''
values_to_replace_bars = [
    'Snack Bar', 'Protein Bar', 'Protein bar', 'Nutrition Bar',
    'Nutrition bar', 'Muesli Bar', 'Millet Bar', 'Milk Compound Bar',
    'Immunity Bar', 'Granola Bar', 'Fruit Bar', 'Fiber Bar',
    'Energy Bar', 'Cereal Bar', 'Multigrain Baked Bar',
    'Millet Wafer Bar', 'Protein Wafer Bar', 'Fruti Bar'
]
df.loc[df['Sub Category '].isin(values_to_replace_bars), 'Sub Category '] = 'Bars'
'''
# -----------------------------
# Lollipop
# -----------------------------
values_to_replace_lollipop = ['Lollipop', 'LolliPop']
df.loc[df['Sub Category '].isin(values_to_replace_lollipop), 'Sub Category '] = 'Lollipop'

# -----------------------------
# Chikki
# -----------------------------
values_to_replace_chikki = ['Chikki', 'chikki']
df.loc[df['Sub Category '].isin(values_to_replace_chikki), 'Sub Category '] = 'Chikki'

# -----------------------------
# Fruit-based (chews / bites / jelly)
# -----------------------------
values_to_replace_fruit = [
    'Fruit Chews', 'Fruit Chew', 'Fruity Chews', 'Chewy Fruit'
]
df.loc[df['Sub Category '].isin(values_to_replace_fruit), 'Sub Category '] = 'Fruit Chews'

# -----------------------------
# Praline
# -----------------------------
values_to_replace_praline = ['Praline', 'Pralines']
df.loc[df['Sub Category '].isin(values_to_replace_praline), 'Sub Category '] = 'Praline'

# -----------------------------
# Eclairs
# -----------------------------
values_to_replace_eclairs = ['Eclairs']
df.loc[df['Sub Category '].isin(values_to_replace_eclairs), 'Sub Category '] = 'Eclairs'

# -----------------------------
# Misc Cleanup
# -----------------------------
values_to_replace_misc = [
    'Panned chewy Candy', 'Panned Chewy Candy'
]
df.loc[df['Sub Category '].isin(values_to_replace_misc), 'Sub Category '] = 'Candy'


# -----------------------------
# Final Check
# -----------------------------

print(df[df['Category'] == 'Confectionery']['Sub Category '].unique())

['Wafer Chocolate' 'Truffle' 'Toffee' 'Soft Candy' 'Chewy Candy'
 'Snack Bar' 'Protein Bar' 'Protein bar' 'Praline' 'Panned Chocolate'
 'Panned chocolate' 'Nutrition Bar' 'Nutrition bar' 'Muesli Bar'
 'Millet Bar' 'Milk Compound Bar' 'Lollipop' 'Jujubes' 'Jelly'
 'Immunity Bar' 'Hard Boiled' 'Gum' 'Granola Bar' 'Fruit Bar' 'Fiber Bar'
 'Energy Bar' 'Eclairs' 'Chocolate' 'Chikki' 'Bar Chocolate'
 'Millet Wafer Bar' 'Fruit Chews' 'Candy' 'Protein Wafer Bar' 'Donut'
 'Eclairs Toffee' 'Soft Center' 'Eclair Toffee' 'Jelly Candy'
 'Truffle Brownie' 'Fruit Bites' 'Multigrain Baked Bar' 'Fruit Jelly'
 'Cereal Bar' 'Fruti Bar' 'Fruit Bite']


### Bakery

In [37]:
# -----------------------------
# Basic Cleaning First
# -----------------------------
df['Sub Category '] = (
    df['Sub Category ']
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# -----------------------------
# Cake Variants
# -----------------------------
values_to_replace_cake = [
    'Slice Cake', 'slice Cake', 'Slice cake',
]
df.loc[df['Sub Category '].isin(values_to_replace_cake), 'Sub Category '] = 'Slice Cake'

values_to_replace_rcake = [
     'Bar Cake', 'Bar cake'
]
df.loc[df['Sub Category '].isin(values_to_replace_rcake), 'Sub Category '] = 'Bar Cake'
# -----------------------------
# Rusk / Toast Variants
# -----------------------------
values_to_replace_rusk = [
    'Rusk / Toast', 'Rusk / toast', 'Rusk/ Toast',
    'Rusk /Toast', 'Rusk / Toasr', 'Rusk/ Toat'
]
df.loc[df['Sub Category '].isin(values_to_replace_rusk), 'Sub Category '] = 'Rusk / Toast'

# -----------------------------
# Waffle Variants
# -----------------------------
values_to_replace_waffle = ['Waffles', 'Waffle']
df.loc[df['Sub Category '].isin(values_to_replace_waffle), 'Sub Category '] = 'Waffles'

# -----------------------------
# Pudding Variants
# -----------------------------
values_to_replace_pudding = ['Pudding', 'pudding']
df.loc[df['Sub Category '].isin(values_to_replace_pudding), 'Sub Category '] = 'Pudding'

# -----------------------------
# Brownie Variants
# -----------------------------
values_to_replace_brownie = ['Brownie', 'Sandwich Brownie']
df.loc[df['Sub Category '].isin(values_to_replace_brownie), 'Sub Category '] = 'Brownie'

# -----------------------------
# Swiss Roll Variants (fix typo)
# -----------------------------
values_to_replace_swiss_roll = ['Swiss Roll', 'Swill Roll']
df.loc[df['Sub Category '].isin(values_to_replace_swiss_roll), 'Sub Category '] = 'Swiss Roll'

# -----------------------------
# Muffins / Cupcakes
# -----------------------------
values_to_replace_muffin = ['Muffins / Cup Cakes']
df.loc[df['Sub Category '].isin(values_to_replace_muffin), 'Sub Category '] = 'Muffin / Cupcake'



# -----------------------------
# Final Check
# -----------------------------
print(df[df['Category'] == 'Bakery (Rusk/Toast/Cakes/Khari)']['Sub Category '].unique())

['Waffles' 'Swiss Roll' 'Cake' 'Slice Cake' 'Rusk / Toast' 'Roll Cake'
 'Quick Bread' 'Pudding' 'Pie' 'Muffin/Cup Cakes' 'Layer Cake' 'Khari'
 'Donut' 'Croissant' 'Cream Roll' 'Bun' 'Brownie' 'Bar Cake' '']


### Misc

In [38]:
df.to_excel("a112b.xlsx")

In [39]:
df.to_csv("a112c.csv")